# Session 05: From documents to clean text

You build a readiness table for 50 of your own documents on a sheet with six named columns. The team notebook builds the same table for all 121 Kitara documents so you can compare.

**SOLUTION VERSION. Tutor only.** This is the team notebook. Leaders can follow it on the shared screen. Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("customers.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith(".csv")), "and the docs folder")

## Task 1. Find the scans
Load `docs_metadata.csv`. For every PDF, read page one with pypdf and treat fewer than 40 characters as no text layer. Add a column `text_layer` (True or False; non-PDF files count as True) and keep the rows with no text layer in `no_layer`.

In [ ]:
import pandas as pd
from pypdf import PdfReader

meta = pd.read_csv('docs_metadata.csv', dtype=str, keep_default_na=False)

def has_text_layer(f):
    if not f.endswith('.pdf'):
        return True
    return len((PdfReader('docs/' + f).pages[0].extract_text() or '').strip()) >= 40

meta['text_layer'] = meta.file.map(has_text_layer)
no_layer = meta[~meta.text_layer]
print(len(no_layer))

In [ ]:
assert len(no_layer) == 8, 'There should be 8 documents with no text layer'
assert set(no_layer.format) == {'scanned_pdf'}, 'Only scanned PDFs should be in no_layer'
print('Task 1 correct')

## Task 2. How well does OCR read each scan?
The first cell installs Tesseract if the notebook does not have it. Then compute Tesseract's mean word confidence for each scan in `no_layer` into a dictionary `conf`, and list the files under 85 in `poor`.

In [ ]:
import shutil, subprocess
if shutil.which('tesseract') is None:
    subprocess.run('apt-get install -y -q tesseract-ocr poppler-utils && pip install -q pytesseract', shell=True)

In [ ]:
import glob, tempfile
import pytesseract
from PIL import Image

def mean_confidence(pdf):
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(['pdftoppm', '-r', '150', '-png', pdf, tmp + '/p'], check=True)
        page = Image.open(sorted(glob.glob(tmp + '/p*.png'))[0])
        d = pytesseract.image_to_data(page, output_type=pytesseract.Output.DICT)
    scores = [float(c) for c, t in zip(d['conf'], d['text']) if t.strip() and float(c) >= 0]
    return sum(scores) / len(scores)

conf = {f: mean_confidence('docs/' + f) for f in no_layer.file}
poor = sorted(f for f, c in conf.items() if c < 85)
print(poor)

In [ ]:
assert len(conf) == 8, 'conf should hold all 8 scans'
assert len(poor) == 3, 'Three scans should fall under 85'
assert all('2018' in f or '2015' in f or '2017' in f for f in poor), 'The poor scans are the 2018, 2017 and 2015 ones'
print('Task 2 correct')

## Task 3. Which old documents say nothing about being old?
Load the whole corpus with `ks.load_corpus('.')` (it runs OCR on the scans and takes about twenty seconds). Keep the superseded documents in `old`. Then keep in `silent` those whose text has none of the words supersed, replaces, replaced, withdrawn or no longer.

In [ ]:
import kitara_search as ks

corpus = ks.load_corpus('.')
notice = corpus.text.str.contains('supersed|replaces|replaced|withdrawn|no longer', case=False)
old = corpus[corpus.status == 'superseded']
silent = old[~notice[old.index]]
print(len(old), len(silent))

In [ ]:
assert len(old) == 11, 'There are 11 superseded documents'
assert len(silent) == 9, 'Nine superseded documents never mention it in their text'
print('Task 3 correct')

## Task 4. Chunk plan
Add a column `n_chunks` to `corpus` with the number of chunks `ks.chunk_text(text, 200, 40)` gives for each document. Store the total in `total_chunks` and the number of documents with more than one chunk in `multi`.

In [ ]:
corpus['n_chunks'] = corpus.text.map(lambda t: len(ks.chunk_text(t, 200, 40)))
total_chunks = int(corpus.n_chunks.sum())
multi = int((corpus.n_chunks > 1).sum())
print(total_chunks, multi)

In [ ]:
assert total_chunks == 139, 'The 121 documents should give 139 chunks'
assert multi == 16, 'Sixteen documents need more than one chunk'
print('Task 4 correct')

## Task 5. The readiness table
Build `table` with one row per document and the columns `file`, `format`, `text_quality` (scan poor, scan ok, table or text ok), `version_status`, `sensitivity`, `chunk_plan` and `verdict`. Use `conf` for the scans and `pdfplumber`'s `find_tables()` for the table PDFs. The verdict is repair first for a poor scan or a table, filter first for a superseded or confidential document or one with personal data, and ready otherwise.

In [ ]:
import pdfplumber

has_layer = meta.set_index('file').text_layer

def tables_in(f):
    if not f.endswith('.pdf') or not has_layer[f]:
        return 0
    with pdfplumber.open('docs/' + f) as pdf:
        return sum(len(p.find_tables()) for p in pdf.pages)

def quality(f):
    if f in conf:
        return 'scan poor' if conf[f] < 85 else 'scan ok'
    return 'table' if tables_in(f) else 'text ok'

table = pd.DataFrame({'file': corpus.file, 'format': corpus.format})
table['text_quality'] = corpus.file.map(quality)
table['version_status'] = corpus.status
table['sensitivity'] = corpus.sensitivity
table['chunk_plan'] = [('repair first' if q == 'scan poor' else 'table rows' if q == 'table'
                        else 'whole document' if n == 1 else 'fixed 200 / 40')
                       for q, n in zip(table.text_quality, corpus.n_chunks)]

def verdict(row, personal):
    if row.text_quality in ('scan poor', 'table'):
        return 'repair first'
    if row.version_status == 'superseded' or row.sensitivity == 'confidential' or personal == 'Y':
        return 'filter first'
    return 'ready'

table['verdict'] = [verdict(r, p) for r, p in zip(table.itertuples(), corpus.contains_personal_data)]
print(table.verdict.value_counts().to_dict())

In [ ]:
vc = table.verdict.value_counts().to_dict()
assert vc == {'ready': 70, 'filter first': 46, 'repair first': 5}, f'Expected 70 ready, 46 filter first, 5 repair first. Got {vc}'
assert (table.text_quality == 'table').sum() == 2, 'Two PDFs hold tables'
print('Task 5 correct')

## Question for the leader
Of 121 Kitara documents, 70 are ready, 46 need a filter or label first and 5 need repair. Which group would you fix first before paying for a search pilot, who in your organisation owns that fix, and what share of your own documents must be ready before you start? Write your answer in two sentences.